In [1]:
!pip install -q langchain langchain-groq


In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")

Groq connected successfully!


In [9]:
SYSTEM = """
ROLE:
You are a friendly and helpful assistant for a Grocery Store.

BUSINESS:
We sell groceries and provide local delivery services.

FACTS:
- Basmati rice: Rs 120 per kg
- Wheat: Rs 40 per kg
- Sugar: Rs 45 per kg
- Apples: Rs 100 per kg
- Sunflower oil: Rs 150 per litre

RULES:
- Answer customers politely.
- Give prices only from the provided catalogue.
- Never invent product prices or availability.
- If a product is not listed, say "I'll check with the owner."
- Ask for the customer's location before discussing delivery confirmation.
- Never promise unconfirmed delivery times.
-- If delivery coverage or delivery time is not confirmed in the provided facts, reply exactly: "I'll check with the owner."

FORMAT:
Keep answers short, friendly and easy to understand.
Match the customer's language whenever possible.
"""

print(SYSTEM)


ROLE:
You are a friendly and helpful assistant for a Grocery Store.

BUSINESS:
We sell groceries and provide local delivery services.

FACTS:
- Basmati rice: Rs 120 per kg
- Wheat: Rs 40 per kg
- Sugar: Rs 45 per kg
- Apples: Rs 100 per kg
- Sunflower oil: Rs 150 per litre

RULES:
- Answer customers politely.
- Give prices only from the provided catalogue.
- Never invent product prices or availability.
- If a product is not listed, say "I'll check with the owner."
- Ask for the customer's location before discussing delivery confirmation.
- Never promise unconfirmed delivery times.
-- If delivery coverage or delivery time is not confirmed in the provided facts, reply exactly: "I'll check with the owner."

FORMAT:
Keep answers short, friendly and easy to understand.
Match the customer's language whenever possible.



In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print("Grocery Store Assistant is ready!")

Grocery Store Assistant is ready!


In [5]:
print(ask("What is the price of 2 kg basmati rice?"))

Sure! 2 kg of basmati rice would cost Rs 240.


In [10]:
import csv

test_data = [
    ["question", "must_include"],
    ["What is the price of basmati rice?", "120"],
    ["How much does wheat cost per kg?", "40"],
    ["What is the price of sugar?", "45"],
    ["How much are apples per kg?", "100"],
    ["What is the price of sunflower oil?", "150"],
    ["What is the price of 2 kg basmati rice?", "240"],
    ["Do you sell milk?", "check with the owner"],
    ["What is the price of chicken?", "check with the owner"],
    ["Can you deliver to Kakinada?", "check with the owner"],
    ["Can you guarantee delivery within one hour?", "check with the owner"]
]

with open("tests.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(test_data)

print("tests.csv created successfully!")

tests.csv created successfully!


In [11]:
import csv

def run_tests():
    passed = 0
    total = 0

    with open("tests.csv", "r") as file:
        rows = csv.DictReader(file)

        for row in rows:
            question = row["question"]
            expected = row["must_include"]

            answer = chain.invoke({
                "history": [],
                "question": question
            })

            if expected.lower() in answer.lower():
                print("PASS:", question)
                passed += 1
            else:
                print("FAIL:", question)

            total += 1

    print("\nTotal tests:", total)
    print("Passed:", passed)
    print("Pass rate:", round(passed / total * 100, 1), "%")

run_tests()

PASS: What is the price of basmati rice?
PASS: How much does wheat cost per kg?
PASS: What is the price of sugar?
PASS: How much are apples per kg?
PASS: What is the price of sunflower oil?
PASS: What is the price of 2 kg basmati rice?
PASS: Do you sell milk?
PASS: What is the price of chicken?
FAIL: Can you deliver to Kakinada?
FAIL: Can you guarantee delivery within one hour?

Total tests: 10
Passed: 8
Pass rate: 80.0 %


In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

print("Updated Grocery Store Assistant is ready!")

Updated Grocery Store Assistant is ready!


In [13]:
run_tests()

PASS: What is the price of basmati rice?
PASS: How much does wheat cost per kg?
PASS: What is the price of sugar?
PASS: How much are apples per kg?
PASS: What is the price of sunflower oil?
PASS: What is the price of 2 kg basmati rice?
PASS: Do you sell milk?
PASS: What is the price of chicken?
PASS: Can you deliver to Kakinada?
PASS: Can you guarantee delivery within one hour?

Total tests: 10
Passed: 10
Pass rate: 100.0 %


Task 3 – Prompt Improvement

Original pass rate: 80%
Improved pass rate: 100%

Improvement made:
Added a rule to reply exactly
"I'll check with the owner."
when delivery coverage or delivery time
is not confirmed.

Result:
The pass rate increased from 80% to 100%.

In [14]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("Temperature changed to 1.0!")

Temperature changed to 1.0!


In [15]:
print("TEMPERATURE 1.0 - RUN 1")
run_tests()

TEMPERATURE 1.0 - RUN 1
PASS: What is the price of basmati rice?
PASS: How much does wheat cost per kg?
PASS: What is the price of sugar?
PASS: How much are apples per kg?
PASS: What is the price of sunflower oil?
PASS: What is the price of 2 kg basmati rice?
PASS: Do you sell milk?
PASS: What is the price of chicken?
PASS: Can you deliver to Kakinada?
PASS: Can you guarantee delivery within one hour?

Total tests: 10
Passed: 10
Pass rate: 100.0 %


In [16]:
print("TEMPERATURE 1.0 - RUN 2")
run_tests()

TEMPERATURE 1.0 - RUN 2
PASS: What is the price of basmati rice?
PASS: How much does wheat cost per kg?
PASS: What is the price of sugar?
PASS: How much are apples per kg?
PASS: What is the price of sunflower oil?
PASS: What is the price of 2 kg basmati rice?
PASS: Do you sell milk?
PASS: What is the price of chicken?
PASS: Can you deliver to Kakinada?
PASS: Can you guarantee delivery within one hour?

Total tests: 10
Passed: 10
Pass rate: 100.0 %


Task 4 – Temperature Experiment

Temperature: 1.0

Run 1:
Passed: 10/10
Pass rate: 100%

Run 2:
Passed: 10/10
Pass rate: 100%

Observation:
Both runs achieved a 100% pass rate.
The PASS/FAIL results did not change between runs.

Note:
The test status remained the same. Response wording may still vary.

## Task 5 – Agent or Not?

### 1. Calculate the total grocery bill
- **Job:** Calculate the total price based on product prices and quantities.
- **Category:** Plain Code
- **Reason:** Mathematical calculations can be performed using Python without AI.

### 2. Answer customer questions about grocery prices
- **Job:** Answer customer questions about products and their prices using the grocery catalogue.
- **Category:** One LLM Call
- **Reason:** An LLM can understand the customer's question and generate a suitable response in one call.

### 3. Handle complete customer orders
- **Job:** Collect customer details, understand the order, check product availability, calculate the bill, verify delivery details, and save the order.
- **Category:** Agent
- **Reason:** It requires multiple steps, tools, and decisions to complete the customer's order.